# ols_engine notebook version


In [1]:
"""
ols_engine notebook — OLS + HC3 robust standard errors via SVD.
Dependencies: numpy, scipy only. No statsmodels.
"""

import numpy as np
from scipy import stats


def encode_categoricals(df, cat_cols, drop_first=True):
    """One-hot encode categorical columns. Returns (df_copy, list_of_dummy_col_names)."""
    df = df.copy()
    dummy_cols = []
    for col in cat_cols:
        raw = df[col].astype(str)
        clean = raw.str.replace(r"[^A-Za-z0-9]", "_", regex=True).str.strip("_")
        levels = sorted(clean.unique())
        if drop_first and levels:
            levels = levels[1:]  # drop reference level (alphabetically first)
        for lv in levels:
            cname = f"{col}__{lv}"
            df[cname] = (clean == lv).astype(float)
            dummy_cols.append(cname)
    return df, dummy_cols


class OLSResult:
    """Container for all OLS output."""

    def __init__(
        self,
        *,
        params,
        bse,
        tvalues,
        pvalues,
        conf_int,
        rsquared,
        rsquared_adj,
        nobs,
        df_resid,
        df_model,
        fvalue,
        f_pvalue,
        mse_resid,
        fitted,
        resid,
        feature_names,
        condition_number,
    ):
        self.params = dict(zip(feature_names, params))
        self.bse = dict(zip(feature_names, bse))
        self.tvalues = dict(zip(feature_names, tvalues))
        self.pvalues = dict(zip(feature_names, pvalues))
        self.conf_int = {
            n: (lo, hi) for n, lo, hi in zip(feature_names, conf_int[:, 0], conf_int[:, 1])
        }
        self.rsquared = rsquared
        self.rsquared_adj = rsquared_adj
        self.nobs = nobs
        self.df_resid = df_resid
        self.df_model = df_model
        self.fvalue = fvalue
        self.f_pvalue = f_pvalue
        self.mse_resid = mse_resid
        self.fittedvalues = fitted
        self.resid = resid
        self.feature_names = feature_names
        self.condition_number = condition_number

    def summary_non_fe(self, fe_prefixes=("Comp__", "Team__", "Season__")):
        """Return dict of non-FE coefficients."""
        return {
            v: {
                "coef": self.params[v],
                "se": self.bse[v],
                "t": self.tvalues[v],
                "p": self.pvalues[v],
                "ci_lo": self.conf_int[v][0],
                "ci_hi": self.conf_int[v][1],
            }
            for v in self.feature_names
            if not any(v.startswith(p) for p in fe_prefixes)
        }


def ols_hc3(X_df, y, alpha=0.05):
    """
    OLS with HC3 heteroskedasticity-robust standard errors.
    Uses SVD of X for numerical stability.
    X_df must include column 'Intercept' as first column.
    """
    X = X_df.values.astype(float)
    names = list(X_df.columns)
    y = np.asarray(y, dtype=float)

    bad = ~(np.isfinite(X).all(axis=1) & np.isfinite(y))
    if bad.any():
        X = X[~bad]
        y = y[~bad]

    n, k = X.shape
    U, S, Vt = np.linalg.svd(X, full_matrices=False)
    tol = 1e-9 * S[0]
    rank = int((S > tol).sum())
    condition_number = float(S[0] / S[rank - 1]) if rank > 0 else np.inf

    proj = U[:, :rank].T @ y
    beta = Vt.T @ np.concatenate([proj / S[:rank], np.zeros(k - rank)])

    fitted = X @ beta
    resid = y - fitted
    assert abs(resid.mean()) < 1e-8, (
        f"Non-zero mean residual {resid.mean():.2e} — intercept missing or data issue"
    )

    df_resid = max(n - rank, 1)
    df_model = max(rank - 1, 1)
    ss_tot = np.sum((y - y.mean()) ** 2)
    ss_res = np.sum(resid ** 2)
    r2 = 1 - ss_res / ss_tot if ss_tot > 0 else 0.0
    r2_adj = 1 - (ss_res / df_resid) / (ss_tot / (n - 1)) if n > 1 and ss_tot > 0 else 0.0
    mse = ss_res / df_resid

    h = np.sum(U[:, :rank] ** 2, axis=1)
    h = np.clip(h, 0, 1 - 1e-10)
    e_adj = resid / (1 - h)

    XtX_pinv = Vt[:rank].T @ np.diag(1 / (S[:rank] ** 2)) @ Vt[:rank]
    meat = (X * e_adj[:, None]).T @ (X * e_adj[:, None])
    V_hc3 = XtX_pinv @ meat @ XtX_pinv
    bse = np.sqrt(np.maximum(np.diag(V_hc3), 0))

    tv = np.where(bse > 0, beta / bse, 0.0)
    pv = 2 * stats.t.sf(np.abs(tv), df=df_resid)
    tcrit = stats.t.ppf(1 - alpha / 2, df=df_resid)
    ci = np.column_stack([beta - tcrit * bse, beta + tcrit * bse])

    fv = max((r2 / df_model) / ((1 - r2) / df_resid), 0) if r2 < 1 else np.inf
    fp = stats.f.sf(fv, df_model, df_resid)

    return OLSResult(
        params=beta,
        bse=bse,
        tvalues=tv,
        pvalues=pv,
        conf_int=ci,
        rsquared=r2,
        rsquared_adj=r2_adj,
        nobs=n,
        df_resid=df_resid,
        df_model=df_model,
        fvalue=fv,
        f_pvalue=fp,
        mse_resid=mse,
        fitted=fitted,
        resid=resid,
        feature_names=names,
        condition_number=condition_number,
    )


In [2]:
# Quick smoke test so output is visible in the notebook
import pandas as pd

np.random.seed(7)
n = 40
x1 = np.random.normal(size=n)
x2 = np.random.normal(size=n)
y = 1.5 + 2.0 * x1 - 0.7 * x2 + np.random.normal(scale=0.3, size=n)

X = pd.DataFrame({
    "Intercept": 1.0,
    "x1": x1,
    "x2": x2,
})
res = ols_hc3(X, y)
print("OLS smoke test OK")
print("nobs:", res.nobs)
print("R2:", round(res.rsquared, 4))
print("beta_x1:", round(res.params["x1"], 4))
print("beta_x2:", round(res.params["x2"], 4))

OLS smoke test OK
nobs: 40
R2: 0.9881
beta_x1: 1.9952
beta_x2: -0.7379
